<a target="_blank" href="https://colab.research.google.com/github/adamya-singh/machine-learning-exercises/blob/master/neuroscope-neel-nanda/Interactive_Neuroscope.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Interactive Neuroscope

*This is an interactive accompaniment to [neuroscope.io](https://neuroscope.io) and to the [studying learned language features post](https://www.alignmentforum.org/posts/Qup9gorqpd9qKAEav/200-cop-in-mi-studying-learned-features-in-language-models) in [200 Concrete Open Problems in Mechanistic Interpretability](https://neelnanda.io/concrete-open-problems)*

There's a surprisingly rich ecosystem of easy ways to create interactive graphics, especially for ML systems. If you're trying to do mechanistic interpretability, the ability to do web dev and to both visualize data and interact with it seems high value!

This version swaps the original dense GPT-2 neuron viewer for an OLMoE expert-routing viewer. The goal is still to show what quickly hacking together a custom visualisation can look like, while keeping the code simple enough to edit in a notebook.

Note that you'll need to run the code yourself to get the interactive interface, so the cell at the bottom will be blank at first!

To emphasise - the point of this notebook is to be a rough proof of concept that just about works, *not* to be the well executed ideal of interactively studying expert routing.

## Setup

In [23]:
# NBVAL_IGNORE_OUTPUT
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

DEVELOPMENT_MODE = True
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except:
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    # Code to automatically update notebook imports as they are edited without restarting the kernel
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install "transformers>=4.57.0"
    %pip install torch
    %pip install gradio
    %pip install sentencepiece


Running as a Colab notebook


In [24]:
# NBVAL_IGNORE_OUTPUT
import html

import gradio as gr
import torch
from IPython.display import HTML, display
from transformers import AutoTokenizer, OlmoeForCausalLM


## Extracting Router Probabilities

We first write some code to extract OLMoE router probabilities for a given layer and expert, for a given text.

In [25]:
# NBVAL_IGNORE_OUTPUT
model_name = "allenai/OLMoE-1B-7B-0924"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = OlmoeForCausalLM.from_pretrained(model_name, dtype="auto").to(device)
model.eval()
print(f"Loaded {model_name} on {device}")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded allenai/OLMoE-1B-7B-0924 on cuda


In [26]:
def get_router_probs(text, layer, expert_index):
    if layer is None:
        raise ValueError("Please select a layer.")
    if expert_index is None:
        raise ValueError("Please select an expert.")

    layer = int(layer)
    expert_index = int(expert_index)
    num_layers = model.config.num_hidden_layers
    num_experts = model.config.num_experts

    if not 0 <= layer < num_layers:
        raise ValueError(f"Layer must be between 0 and {num_layers - 1}.")
    if not 0 <= expert_index < num_experts:
        raise ValueError(f"Expert index must be between 0 and {num_experts - 1}.")

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True)

    router_logits = outputs.router_logits
    if router_logits is None:
        raise ValueError("Model did not return router logits.")

    layer_router_logits = router_logits[layer]
    if layer_router_logits.dim() == 2:
        batch_size, seq_len = inputs["input_ids"].shape
        layer_router_logits = layer_router_logits.view(batch_size, seq_len, -1)
    elif layer_router_logits.dim() != 3:
        raise ValueError(
            f"Unexpected router logits shape for layer {layer}: {tuple(layer_router_logits.shape)}"
        )

    router_probs = torch.softmax(layer_router_logits.float(), dim=-1)[0, :, expert_index]
    input_ids = inputs["input_ids"][0].detach().cpu().tolist()
    str_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    return router_probs.detach().to(torch.float32).cpu().numpy(), str_tokens


We can run this function and verify that it gives vaguely sensible router probabilities.

In [27]:
default_layer = 0
default_expert_index = 0
default_text = "The Apollo program landed astronauts on the moon in 1969."
print(tokenizer.convert_ids_to_tokens(tokenizer(default_text, return_tensors="pt")["input_ids"][0]))


['The', 'ĠApollo', 'Ġprogram', 'Ġlanded', 'Ġastronaut', 's', 'Ġon', 'Ġthe', 'Ġmoon', 'Ġin', 'Ġ1969', '.']


In [28]:
# NBVAL_IGNORE_OUTPUT
print(get_router_probs(default_text, default_layer, default_expert_index)[0])


[0.02316565 0.01555351 0.00413193 0.00677135 0.01038953 0.00258482
 0.00377969 0.00804964 0.00632574 0.00553454 0.00895129 0.00588677]


## Visualizing Router Probabilities

We now write some code to visualize router probabilities on some text. We'll keep the same simple HTML approach as the original notebook, but now each token is colored according to the selected expert's routing probability at the selected layer.

Because these values are probabilities, they live in `[0, 1]`. In practice the colors are still sensitive to `max_val` and `min_val`, so keeping explicit controls is useful while exploring.

In [29]:
# This is some CSS to give each token a thin gray border and preserve whitespace.
style_string = """<style>
    span.token {
        border: 1px solid rgb(123, 123, 123);
        white-space: pre-wrap;
    }
    </style>"""


def calculate_color(val, max_val, min_val):
    # Normalize to [0, 1] and interpolate between slightly off-white and red.
    denom = max(max_val - min_val, 1e-8)
    normalized_val = min(max((val - min_val) / denom, 0.0), 1.0)
    return f"rgb(240, {240 * (1 - normalized_val)}, {240 * (1 - normalized_val)})"


def basic_router_vis(text, layer, expert_index, max_val=None, min_val=None):
    """
    text: The text to visualize
    layer: The layer index
    expert_index: The expert index
    max_val: The top end of our probability range, defaults to the maximum probability
    min_val: The bottom end of our probability range, defaults to the minimum probability

    Returns a string of HTML that displays the text with each token colored according to router probability.
    """
    try:
        probs, str_tokens = get_router_probs(text, layer, expert_index)
    except Exception as error:
        return str(error)

    prob_max = probs.max()
    prob_min = probs.min()
    if max_val is None:
        max_val = prob_max
    if min_val is None:
        min_val = prob_min

    htmls = [style_string]
    htmls.append(f"<h4>Layer: <b>{int(layer)}</b>. Expert Index: <b>{int(expert_index)}</b></h4>")
    htmls.append(
        f"<h4>Display Range: <b>{max_val:.4f}</b>. Min Range: <b>{min_val:.4f}</b></h4>"
    )
    if prob_max != max_val or prob_min != min_val:
        htmls.append(
            f"<h4>Actual Probabilities: <b>{prob_min:.4f}</b> to <b>{prob_max:.4f}</b></h4>"
        )
    for tok, prob in zip(str_tokens, probs):
        safe_tok = html.escape(tok)
        htmls.append(
            f"<span class='token' style='background-color:{calculate_color(prob, max_val, min_val)}' >{safe_tok}</span>"
        )

    return "".join(htmls)


In [30]:
# NBVAL_IGNORE_OUTPUT
# The function outputs a string of HTML
default_max_val = 1.0
default_min_val = 0.0
default_html_string = basic_router_vis(
    default_text,
    default_layer,
    default_expert_index,
    max_val=default_max_val,
    min_val=default_min_val,
)

# IPython lets us display HTML
print("Displayed HTML")
display(HTML(default_html_string))

# We can also print the string directly
print("HTML String - it's just raw HTML code!")
print(default_html_string)


Displayed HTML


HTML String - it's just raw HTML code!
<style>
    span.token {
        border: 1px solid rgb(123, 123, 123);
        white-space: pre-wrap;
    }
    </style><h4>Layer: <b>0</b>. Expert Index: <b>0</b></h4><h4>Display Range: <b>1.0000</b>. Min Range: <b>0.0000</b></h4><h4>Actual Probabilities: <b>0.0026</b> to <b>0.0232</b></h4><span class='token' style='background-color:rgb(240, 234.4402438402176, 234.4402438402176)' >The</span><span class='token' style='background-color:rgb(240, 236.2671571969986, 236.2671571969986)' >ĠApollo</span><span class='token' style='background-color:rgb(240, 239.00833792984486, 239.00833792984486)' >Ġprogram</span><span class='token' style='background-color:rgb(240, 238.37487559765577, 238.37487559765577)' >Ġlanded</span><span class='token' style='background-color:rgb(240, 237.5065116584301, 237.5065116584301)' >Ġastronaut</span><span class='token' style='background-color:rgb(240, 239.37964310869575, 239.37964310869575)' >s</span><span class='token' style='

## Create Interactive UI

We now put all these together to create an interactive visualization in Gradio.

The internal format is that there's a bunch of elements - Textboxes, Numbers, etc which the user can interact with and which return strings and numbers. And we can also define output elements that just display things - in this case, one which takes in an arbitrary HTML string. We call `input.change(update_function, inputs, output)` - this says "if that input element changes, run the update function on the value of each of the elements in `inputs` and set the value of `output` to the output of the function". As a bonus, this gives us live interactivity.

In [31]:
# The `with gr.Blocks() as demo:` syntax just creates a variable called demo containing all these components
with gr.Blocks() as demo:
    gr.HTML(value=f"Hacky Interactive OLMoE Router Viewer for {model_name}")
    # The input elements
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(label="Text", value=default_text)
            # Precision=0 makes it an int, otherwise it's a float
            # Value sets the initial default value
            layer = gr.Number(label="Layer", value=default_layer, precision=0)
            expert_index = gr.Number(
                label="Expert Index", value=default_expert_index, precision=0
            )
            # If empty, these two map to None
            max_val = gr.Number(label="Max Value", value=default_max_val)
            min_val = gr.Number(label="Min Value", value=default_min_val)
            inputs = [text, layer, expert_index, max_val, min_val]
        with gr.Column():
            # The output element
            out = gr.HTML(label="Router Probabilities", value=default_html_string)
    for inp in inputs:
        inp.change(basic_router_vis, inputs, out)


We can now launch our demo element, and we're done! The setting share=True even gives you a public link to the demo (though it just redirects to the backend run by this notebook, and will go away once you turn the notebook off!). Sharing makes it much slower, and can be turned off if you aren't in Colab.

**Exercise:** Explore which tokens different experts at different layers assign high router probability to. Are some experts clearly more active on dates, names, or topic-specific words than others?

In [32]:
# NBVAL_IGNORE_OUTPUT
demo.launch(share=True, height=1000)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f82eb603779d17361.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
